# Demo — Onde o vetor mora: Azure DocumentDB

**Objetivo:** guardar os trechos do acervo, com os vetores, numa coleção do **Azure DocumentDB** (compatível com MongoDB) e repetir a busca e o RAG dos outros notebooks. Ilustra os slides 20 a 23: a pergunta que decide onde o vetor mora, a linha **DocumentDB (API Mongo)** da tabela de opções e os algoritmos de índice.

1. **Preparar os vetores** → os mesmos trechos e o mesmo embedding das outras bases da série;
2. **Criar a estrutura vetorial** → um índice `cosmosSearch` na coleção, com um dos algoritmos do slide 23;
3. **Gravar os trechos** → cada trecho vira um documento, e o vetor é só mais um campo dele;
4. **Buscar por similaridade** → o estágio `$search` da agregação, e o quase empate do SKU do slide 14;
5. **RAG completo** → a mesma resposta com citação, com outra base embaixo.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** um cluster **Azure DocumentDB** (o antigo Azure Cosmos DB for MongoDB vCore), com o IP da sua máquina liberado no firewall. O índice padrão deste notebook, IVF, roda em qualquer tier, inclusive no Free: a página do Free tier só deixa de fora HNSW e DiskANN. Esses dois exigem tier **M30 ou superior**, mas o Learn diverge e algumas páginas citam M40: teste antes da aula. A busca vetorial vem incluída no cluster, sem custo à parte, e a documentação atual não pede feature flag para ela.

## 0. Preparação

Este notebook usa a **Azure OpenAI v1 API** (cliente Python `OpenAI`) para os embeddings e a resposta, e o `pymongo`, o mesmo driver de qualquer MongoDB, para falar com o DocumentDB.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`
- `AZURE_DOCUMENTDB_CONNECTION_STRING`: a string de conexão do portal (menu **Connection strings** do cluster), com a sua senha no lugar de `<password>`. O formato é `mongodb+srv://<usuario>:<senha>@<cluster>.mongocluster.cosmos.azure.com/?tls=true&authMechanism=SCRAM-SHA-256&retrywrites=false&maxIdleTimeMS=120000`. O host com `.global.` antes de `mongocluster` funciona igual num cluster sem réplicas.

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook. A string de conexão do DocumentDB leva a senha: trate-a como chave.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "pymongo>=4.8"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
import warnings

from pymongo import MongoClient

AZURE_DOCUMENTDB_CONNECTION_STRING = os.getenv("AZURE_DOCUMENTDB_CONNECTION_STRING") or getpass(
    "AZURE_DOCUMENTDB_CONNECTION_STRING (mongodb+srv://...): "
)

# O pymongo reconhece o host *.cosmos.azure.com e avisa que é um serviço compatível com MongoDB,
# não o MongoDB. O exemplo oficial da Microsoft silencia esse aviso; fazemos o mesmo.
warnings.filterwarnings("ignore", message="You appear to be connected to a CosmosDB cluster")

cliente_mongo = MongoClient(AZURE_DOCUMENTDB_CONNECTION_STRING)
cliente_mongo.admin.command("ping")  # para aqui se a senha, a string ou o firewall estiverem errados

NOME_BANCO, NOME_COLECAO, NOME_INDICE = "cap05", "trechos", "idx_vetor"
banco = cliente_mongo[NOME_BANCO]
colecao = banco[NOME_COLECAO]

print(f"DocumentDB conectado. Banco '{NOME_BANCO}', coleção '{NOME_COLECAO}'.")

### Funções auxiliares

As mesmas dos outros notebooks do Cap. 5 que as usam, sem mudança:

- `carregar_trechos`: um trecho por seção `## ` de cada documento do acervo;
- `texto_para_embedding` e `gerar_embeddings`: título, seção e conteúdo viram um vetor, com o deployment e a dimensão do `.env`;
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu;
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores

In [ ]:
import json

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Os mesmos 50 trechos e os mesmos vetores dos outros notebooks: mesmo chunking, mesmo modelo e mesma dimensão de embedding. É de propósito. Nesta série só a base muda, e o embedding é o contrato entre indexação e consulta (slide 9), que vai junto para qualquer base.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")

---

# Execução 2 — Criar a estrutura vetorial

No DocumentDB não há esquema para declarar: a coleção aceita qualquer documento. Quem diz que o campo `embedding` é um vetor é o **índice**, criado por um comando, `createIndexes`, com a chave `{"embedding": "cosmosSearch"}` e as `cosmosSearchOptions`: o algoritmo (`kind`), a métrica (`similarity`: `COS`, `L2` ou `IP`) e a dimensão, que vem de `EMBEDDING_DIMENSIONS`.

### Os limites desta base (slide 22)

| Serviço | Dimensões | Índices | Reranking | Quando é a escolha certa |
|---|---|---|---|---|
| DocumentDB (API Mongo) | até 16.000 | IVFFlat, HNSW, DiskANN | Não | O time já vive em Mongo |

Os "até 16.000" valem para DiskANN com compressão por product quantization. Sem compressão, o limite é de 2.000 dimensões; HNSW e IVF com meia precisão chegam a 4.000. As 1.536 dimensões do `text-embedding-3-small` cabem nos três sem compressão; as 3.072 do `text-embedding-3-large`, não.

### O algoritmo do índice (slide 23)

| | HNSW | IVFFlat | DiskANN |
|---|---|---|---|
| Como funciona | Grafo em camadas, em memória | Particiona o espaço em listas | Grafo otimizado para SSD |
| Recall | Alto | Médio | Alto |
| Memória | Alta | Baixa | Baixa |
| Dado que muda | Lida bem | Pede reconstrução | Lida bem |
| `kind` no DocumentDB | `vector-hnsw` | `vector-ivf` | `vector-diskann` |
| Tier mínimo | M30¹ | qualquer um, inclusive o Free² | M30¹ |

¹ O Learn diverge, e algumas páginas citam M40: teste antes da aula. ² Segundo a página do Free tier, só HNSW e DiskANN ficam de fora dele.

O trade-off é sempre recall, memória e tempo de indexação. Usamos IVF com uma lista só (`numLists: 1`): para 50 trechos, equivale a uma busca exaustiva, exata, e roda em qualquer tier. A documentação sugere começar assim em demonstrações e, num acervo de verdade, usar cerca de uma lista para cada 1.000 documentos. Com uma lista só, criar o IVF antes da carga, como aqui, não muda nada. Com várias, o IVF agrupa os vetores já gravados: convém criá-lo depois da carga e recriá-lo quando o dado mudar muito. É o "pede reconstrução" do slide 23; o Learn só diz do HNSW que ele pode ser criado com a coleção vazia.

Para produção, a Microsoft recomenda DiskANN (M30 ou superior), porque ele deixa a maior parte do índice em disco, longe da memória da carga operacional. No slide 23, ele também lida bem com dado que muda.

Para trocar, mude `TIPO_INDICE` ou um parâmetro e rode esta célula de novo. Um campo aceita um único índice vetorial: se a definição registrada for outra, o código apaga o índice antigo e cria o novo sobre os documentos já gravados, que ficam. É também o jeito de ver o IVF com várias listas: depois da primeira carga, mude `numLists` e rode esta célula de novo. Se o cluster recusar o índice novo, como faz com HNSW e DiskANN abaixo do M30, o código recria o antigo e para com um aviso: a busca continua com o índice de antes. Mudou a dimensão do embedding no `.env`? Reinicie o kernel, para o valor novo valer: os vetores gravados têm o tamanho antigo, e a célula para com um aviso. Rode a limpeza do fim do notebook e depois o notebook inteiro, com `APAGAR_RECURSOS` de volta em `False`: é o "reindexar tudo" do slide 10.

In [ ]:
from pymongo.errors import OperationFailure

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

# Free tier, M10, M20 e M25: só "vector-ivf". M30 ou superior (M40, em algumas páginas do Learn):
# também "vector-hnsw" e "vector-diskann".
TIPO_INDICE = "vector-ivf"
PARAMETROS_DO_ALGORITMO = {
    "vector-ivf": {"numLists": 1},                      # 1 lista: busca exaustiva, a sugestão da doc para demos
    "vector-hnsw": {"m": 16, "efConstruction": 64},     # valores padrão do serviço
    "vector-diskann": {"maxDegree": 32, "lBuild": 50},  # valores padrão do serviço
}
opcoes_vetoriais = {
    "kind": TIPO_INDICE,
    "similarity": "COS",  # cosseno; as outras métricas são L2 (euclidiana) e IP (produto interno)
    "dimensions": EMBEDDING_DIMENSIONS,
    **PARAMETROS_DO_ALGORITMO[TIPO_INDICE],
}


def criar_indice(opcoes: dict) -> None:
    banco.command({
        "createIndexes": NOME_COLECAO,
        "indexes": [{"name": NOME_INDICE, "key": {"embedding": "cosmosSearch"}, "cosmosSearchOptions": opcoes}],
    })


if NOME_COLECAO not in banco.list_collection_names():
    banco.create_collection(NOME_COLECAO)

indice_atual = next((i for i in colecao.list_indexes() if i["name"] == NOME_INDICE), None)
opcoes_anteriores = None  # a definição apagada, para recriá-la se o cluster recusar a nova
if indice_atual:
    # O listIndexes devolve as opções com que o índice foi criado; comparamos as que este notebook define.
    definicao_atual = {opcao: indice_atual["cosmosSearch"].get(opcao) for opcao in opcoes_vetoriais}
    if definicao_atual["dimensions"] != EMBEDDING_DIMENSIONS and colecao.count_documents({}) > 0:
        raise RuntimeError(
            f"A dimensão do embedding mudou ({definicao_atual['dimensions']} → {EMBEDDING_DIMENSIONS}), e os vetores "
            "gravados têm o tamanho antigo. Rode a limpeza (APAGAR_RECURSOS = True), volte a flag para False e "
            "depois rode o notebook inteiro."
        )
    if definicao_atual != opcoes_vetoriais:
        # Mudou o algoritmo ou um parâmetro: o antigo sai, porque o campo só aceita um índice vetorial.
        opcoes_anteriores = dict(indice_atual["cosmosSearch"])
        colecao.drop_index(NOME_INDICE)
        print(f"Índice anterior apagado: {opcoes_anteriores}. Os documentos ficam.")
        indice_atual = None

if indice_atual:
    print(f"O índice '{NOME_INDICE}' já existe com esta definição: nada a criar.")
else:
    try:
        criar_indice(opcoes_vetoriais)
    except OperationFailure as erro:
        if opcoes_anteriores:  # o antigo já tinha saído: ele volta, para a coleção não ficar sem índice vetorial
            criar_indice(opcoes_anteriores)
        raise RuntimeError(
            f"O cluster recusou o índice {TIPO_INDICE} (o motivo está no erro acima). Se for HNSW ou DiskANN, "
            "a causa provável é o tier: os dois exigem M30 ou superior (M40, em algumas páginas do Learn). "
            'Volte TIPO_INDICE para "vector-ivf" ou suba o tier.'
            + (" O índice anterior foi recriado, e a busca continua com ele." if opcoes_anteriores else "")
        ) from erro
    print(f"Índice '{NOME_INDICE}' criado: {TIPO_INDICE}, cosseno, {EMBEDDING_DIMENSIONS} dimensões.")

print("\nÍndices da coleção, como o serviço os registrou:")
display(pd.DataFrame([
    {"índice": i["name"], "chave": i["key"], "opções vetoriais": i.get("cosmosSearch", "")}
    for i in colecao.list_indexes()
]))

---

# Execução 3 — Gravar os trechos

Cada trecho vira um documento da coleção, e o vetor é só mais um campo dele, ao lado do título, da seção e do texto. É o que o slide 21 chama de "o vetor vive junto do dado operacional": não há um segundo sistema para manter sincronizado.

A gravação usa `replace_one` com `upsert=True` e o `id` do trecho como `_id`: se o documento já existe, é substituído; se não existe, é criado. Rodar o notebook de novo não duplica nada.

In [ ]:
novos = substituidos = 0
for trecho, vetor in zip(trechos, vetores):
    documento = {
        "_id": trecho["id"],
        "doc_id": trecho["doc_id"],
        "titulo": trecho["titulo"],
        "secao": trecho["secao"],
        "conteudo": trecho["conteudo"],
        "fonte": trecho["fonte"],
        "embedding": vetor,  # o vetor mora no mesmo documento que o texto
    }
    resultado = colecao.replace_one({"_id": trecho["id"]}, documento, upsert=True)
    if resultado.upserted_id is not None:
        novos += 1
    else:
        substituidos += 1

print(f"Gravados: {novos + substituidos} | novos: {novos} | substituídos: {substituidos}")
print(f"Documentos na coleção: {colecao.count_documents({})} (trechos no acervo: {len(trechos)})")

---

# Execução 4 — Buscar por similaridade

A busca é um estágio da agregação, `$search` com `cosmosSearch`: o vetor da pergunta, o campo (`path`) e quantos vizinhos queremos (`k`). Um `$project` em seguida escolhe os campos e traz o score com `{"$meta": "searchScore"}`. O vetor de cada trecho não volta na resposta.

**Similaridade, sem conversão.** Com `similarity: "COS"`, o `searchScore` já é a similaridade de cosseno: quanto maior, mais próximo (1 = mesma direção). Não há distância para converter. No AI Search (notebook `tipos-de-rag/01`), o score vetorial era `1 / (1 + distância de cosseno)`: a escala muda de uma base para outra; a ordem tende a ser a mesma, porque as duas notas crescem com o cosseno.

Rodamos as duas perguntas do gabarito que separam os dois lados do slide 14: linguagem natural, em que o vetor tende a acertar, e string exata (SKU 48213-B), em que ele tende a tropeçar.

### O que esperar e por que pode variar

- Na pergunta de linguagem natural, a seção "Como pedir o reembolso de uma viagem" tende a vir em 1º.
- No SKU, os três rolamentos (48213-B, C e D) tendem a aparecer juntos no topo, com similaridades quase iguais. **Trocar de base não muda a natureza do vetor:** o embedding e o cosseno são os mesmos do AI Search, então o quase empate do slide 14 continua aqui. Quem resolve isso é a recuperação (híbrido e reranking, em `tipos-de-rag`); a base é uma decisão de operação (slide 21).
- Os números dependem do modelo e da dimensão do embedding. Com IVF de uma lista, a busca é exaustiva; com HNSW ou DiskANN, ela é aproximada, e um vizinho quase empatado pode trocar de lugar.
- Logo depois de criado, o índice pode levar alguns minutos para ficar pronto, e nesse meio-tempo a busca pode voltar vazia. A célula então para com um aviso, em vez de dizer que o trecho certo ficou fora do top 5: espere e rode-a de novo.

In [ ]:
def buscar(pergunta: str, k: int = 5) -> list[dict]:
    """Busca vetorial no DocumentDB: a pergunta vira vetor e o $search devolve os k trechos mais próximos."""
    vetor = gerar_embeddings([pergunta])[0]
    pipeline = [
        {"$search": {"cosmosSearch": {"vector": vetor, "path": "embedding", "k": k}}},
        # Com similarity COS, o searchScore já é a similaridade de cosseno: quanto maior, mais próximo.
        {"$project": {"titulo": 1, "secao": 1, "conteudo": 1, "fonte": 1, "similaridade": {"$meta": "searchScore"}}},
    ]
    return [
        {
            "id": d["_id"],
            "titulo": d["titulo"],
            "secao": d["secao"],
            "conteudo": d["conteudo"],
            "fonte": d["fonte"],
            "similaridade": d["similaridade"],
        }
        for d in colecao.aggregate(pipeline)
    ]


resultados = {}
for pid in ["linguagem-natural", "string-exata"]:
    p = PERGUNTAS[pid]
    top5 = buscar(p["pergunta"], k=5)
    if not top5:  # índice ainda em construção: "fora do top 5" seria um diagnóstico errado
        raise RuntimeError(
            "Nenhum resultado: logo depois de criado, o índice pode levar alguns minutos para ficar pronto. "
            "Espere e rode esta célula de novo."
        )
    resultados[pid] = top5
    lugar = posicao([t["id"] for t in top5], p["trechos_esperados"][0])
    situacao = f"{lugar}º lugar" if lugar else "fora do top 5"
    display(Markdown(f"**{p['pergunta']}** ({p['categoria']})  \nPosição do trecho certo: **{situacao}**"))
    display(pd.DataFrame([
        {
            "posição": i,
            "similaridade": round(t["similaridade"], 4),
            "diferença para o 1º": round(t["similaridade"] - top5[0]["similaridade"], 4),
            "trecho": f"{t['titulo']} — {t['secao']}",
            "é o trecho certo?": "sim" if t["id"] in p["trechos_esperados"][0] else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

### O que observar

- Com o índice pronto e o IVF de uma lista deste notebook, a busca **sempre** devolve `k` resultados, mesmo quando nenhum trecho responde de verdade: no vetor puro não existe "não encontrei". Quando vem menos, o motivo é técnico, não de relevância: o índice ainda em construção (a busca volta vazia, e a célula acima para com um aviso) ou um IVF de várias listas, que por padrão só percorre a lista mais próxima da pergunta (`nProbes` = 1), e ela pode ter menos de `k` trechos.
- No SKU, olhe a coluna **diferença para o 1º**: os irmãos 48213-B, C e D costumam ficar a centésimos, ou milésimos, uns dos outros. O embedding captura o assunto (prazo de entrega de rolamento), não o código.
- Compare com o notebook `tipos-de-rag/01`: os scores são outros, porque a escala é outra, mas a ordem dos trechos tende a ser a mesma. Trocar de base não melhorou nem piorou a recuperação.

---

# Execução 5 — RAG completo

Os 3 primeiros trechos da busca vão para o modelo, que responde citando o `id` de cada trecho usado. A função `responder` é a mesma das outras bases da série e de `tipos-de-rag`: para a geração, não importa em que base o vetor mora.

### O que esperar e por que pode variar

- Se a seção "Como pedir o reembolso de uma viagem" estiver entre os três, a resposta tende a citar o sistema Conecta Despesas, o prazo de dez dias úteis e a aprovação do gestor.
- O texto muda a cada execução. As citações entre colchetes mostram de que trecho veio cada afirmação.

In [ ]:
pid = "linguagem-natural"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados[pid][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração.
- O modelo, o prompt e a função `responder` são os mesmos das outras bases da série. A base mudou; a resposta não tem por que mudar.

### Leitura arquitetural

- **A linha do slide 22:** DocumentDB (API Mongo), até 16.000 dimensões, IVFFlat, HNSW e DiskANN, sem reranking. Quando é a escolha certa: **"O time já vive em Mongo".**
- **A pergunta do slide 21, "o dado muda em tempo real?":** quando o dado operacional já mora em documentos Mongo e muda o tempo todo (catálogo, cadastro, atendimento), o vetor entra como mais um campo, gravado na mesma escrita do dado, sem pipeline de sincronização com um índice externo. E é a resposta mais comum do slide: usar o que a empresa já opera.
- **Quem opera:** o time que já administra o cluster (usuários, regras de firewall, tier). Nenhum componente novo.
- **Quem monitora:** o mesmo time, com as métricas do cluster no Azure Monitor. O índice vetorial disputa memória com a carga operacional: a documentação avisa que um índice HNSW grande pode esgotar a memória do cluster e atrasar as outras operações, e indica o DiskANN, que deixa a maior parte do índice em disco.
- **Quem paga:** o tier do cluster. A busca vetorial não é cobrada à parte, mas HNSW e DiskANN só existem do M30 para cima (M40, segundo algumas páginas do Learn): aqui, o algoritmo do slide 23 também é uma decisão de custo.
- **Híbrido nesta base:** um índice de texto BM25 (`createSearchIndexes`, ainda em gated preview) na mesma coleção, com a fusão RRF feita na aplicação ou num `$unionWith`. Reranking fica de fora, como diz o slide 22. Híbrido e reranking continuam sendo decisões de recuperação (`tipos-de-rag`), não de base.

### Fala sugerida

> “Leia a última coluna, não as do meio.”

Na linha do DocumentDB, a última coluna diz: o time já vive em Mongo. O que decide não é o benchmark, é quem vai operar o componente que você trouxer.

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos 50 trechos e os mesmos vetores dos outros notebooks | O embedding é o contrato: ele vai junto para qualquer base |
| 2 | O índice vetorial é um comando, `createIndexes` com `cosmosSearch`, e o tier decide o algoritmo | Algoritmo (slide 23) e tier são decisões de custo e de operação |
| 3 | O vetor é um campo do documento, gravado com `replace_one` e `upsert`, sem duplicar | Vetor junto do dado operacional, sem sincronização (slide 21) |
| 4 | Com COS, o `searchScore` já é a similaridade; o quase empate do SKU continua | Trocar a base não conserta a recuperação: isso é híbrido e reranking |
| 5 | A mesma resposta com citação, com outra base embaixo | Escolher a base pelo time que vai operá-la, não pelo benchmark |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Azure DocumentDB — repositório vetorial integrado: índices IVF, HNSW e DiskANN e o estágio `$search`:  
  https://learn.microsoft.com/azure/documentdb/vector-search

- Azure DocumentDB — limites do serviço: os tiers de HNSW e DiskANN (M30, e M40 na seção do Free tier):  
  https://learn.microsoft.com/azure/documentdb/limitations

- Azure DocumentDB — busca vetorial com Python:  
  https://learn.microsoft.com/azure/documentdb/quickstart-python-vector-search

- Azure DocumentDB — busca híbrida, BM25 e vetor com fusão RRF:  
  https://learn.microsoft.com/azure/documentdb/full-text-search-hybrid

---

# Limpeza (opcional)

Com `APAGAR_RECURSOS = True`, a célula abaixo apaga a coleção `trechos`, com os 50 documentos e o índice `idx_vetor`. O cluster não foi criado aqui e continua existindo, e cobrando, fora do Free tier. Apague-o no portal quando não precisar mais dele.

In [ ]:
APAGAR_RECURSOS = False  # True apaga a coleção criada por este notebook

if APAGAR_RECURSOS:
    banco.drop_collection(NOME_COLECAO)
    print(f"Coleção '{NOME_COLECAO}' apagada, com o índice '{NOME_INDICE}'.")
else:
    print(f"Nada foi apagado. A coleção '{NOME_COLECAO}' continua no banco '{NOME_BANCO}'.")